In [4]:
import yfinance as yf
print(yf.__version__)

1.7.0


In [4]:
import os
import finnhub
import datetime as dt
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# This will now work without a KeyError
c = finnhub.Client(api_key=os.environ["FINNHUB_API_KEY"])

today = dt.date.today()
rows = c.company_news("AADX", _from=str(today - dt.timedelta(days=14)), to=str(today))
print(len(rows))
if rows:
    print(rows[0])


4
{'category': 'company', 'datetime': 1791876725, 'headline': 'Jim Cramer Likes This Tech Stock, Recommends Sticking With Applied Aerospace', 'id': 142848246, 'image': 'https://cdn.benzinga.com/files/images/story/2026/10/13/Jim-Cramer-At-The-Cable-Show-2010-An-Eve.jpg?width=2048&height=1536', 'related': 'AADX', 'source': 'Benzinga', 'summary': 'Cramer favors Applied Aerospace, Nokia and BlackBerry, questions Ambarella, dislikes food stocks, and says Ameresco may be oversold.', 'url': 'https://finnhub.io/api/news?id=f413d05e558971fa226282f43799d3bd9121ce28b70b314a815e0720cbaa255f'}


In [2]:
   import finagent; print(finagent.__file__)

/Users/sudhakaransrinivasan/Documents/GitHub/multi-agent-financial-analyst/src/finagent/__init__.py


In [8]:
from finagent.agent.loop import make_client
from finagent.workflows.news_ingest import collect_articles
from finagent.workflows.news_classify import make_llm_classifier, classify_articles

clf = make_llm_classifier(make_client())
for t, subject in (("NVDA", "NVDA (NVIDIA Corporation)"),
                   ("AADX", "AADX (Applied Aerospace & Defense, Inc.)")):
    batch = collect_articles(t)
    out = classify_articles(batch.articles, subject, clf)
    print(t, "kept", len(out.items), "irrelevant", out.irrelevant, "failed", out.failed)
    for a, label in out.items: print("  ", label, "|", a.headline)

NVDA kept 0 irrelevant 8 failed 0
AADX kept 2 irrelevant 2 failed 0
   general | Jim Cramer Likes This Tech Stock, Recommends Sticking With Applied Aerospace
   general | 'This SpaceX supplier has a front-row seat to major shifts in warfare and rocket use'- CNBC Pro


In [5]:
from finagent.agent.loop import make_client
from finagent.workflows.news_ingest import collect_articles
from finagent.workflows.news_classify import make_llm_classifier, classify_articles
from finagent.workflows.news_specialists import make_llm_analyzer, build_insights

client = make_client()
for t, subject in (("NVDA", "NVDA (NVIDIA Corporation)"),
                   ("AADX", "AADX (Applied Aerospace & Defense, Inc.)")):
    batch = collect_articles(t)
    cls = classify_articles(batch.articles, subject, make_llm_classifier(client))
    ins = build_insights(cls, subject, make_llm_analyzer(client))
    print(f"\n{t}: kept {len(cls.items)}, irrelevant {cls.irrelevant}, "
          f"classify failed {cls.failed}, specialist failed {ins.failed}")
    for i in ins.insights:
        print(f"  [{i['category']}/{i['sentiment']}] {i['summary']}")


NVDA: kept 0, irrelevant 8, classify failed 0, specialist failed 0

AADX: kept 2, irrelevant 2, classify failed 0, specialist failed 0
  [general/positive] Jim Cramer recommends sticking with Applied Aerospace, favoring it alongside Nokia and BlackBerry, while questioning Ambarella and expressing a dislike for food stocks.
  [general/positive] AADX is a supplier for SpaceX and is positioned to benefit from major shifts in warfare and rocket use, with investors advised to watch its shares.


In [6]:
for a in collect_articles("NVDA").articles:
    print(clf(a, "NVDA (NVIDIA Corporation)"), "|", a.headline, "|", a.summary[:120])

irrelevant | 3 Absurdly Cheap Stocks to Buy Before the End of 2026 | These stocks are priced incredibly low and possess significant upside.
irrelevant | Which Industrials Stock Do Analysts Like Better: Deere or Caterpillar? | Narratives about the two heavy equipment makers have diverged, leaving one a Wall Street favorite.
irrelevant | How Early Taltz and Zepbound Biomarker Synergies At Eli Lilly (LLY) Have Changed Its Investment Story | Eli Lilly recently reported exploratory Phase 3b data from the TOGETHER-PsO trial showing that combining Taltz with Zepb
irrelevant | Why Bumble Stock Skyrocketed Today | The dating app provider is being bought out at a sizable premium.
irrelevant | You Bought a Semiconductor Fund. On $500,000, $87,743 of It Is NVIDIA | A single semiconductor ETF sounds like a spread bet across dozens of chip companies, but the math inside one popular fun
irrelevant | Micron Just Raised Its Buyback Authorization to $35 Billion. Prediction: It Spends More Buying Back St

In [1]:
from finagent.agent.loop import make_client
from finagent.workflows.news_ingest import collect_articles
from finagent.workflows.news_classify import make_llm_classifier, classify_articles
from finagent.workflows.news_specialists import make_llm_analyzer, build_insights

client = make_client()
clf = make_llm_classifier(client)

In [2]:
for a in collect_articles("AADX", company_name="Applied Aerospace & Defense, Inc.").articles:
    if "spacex" in (a.headline + a.summary).lower() or "warfare" in (a.headline + a.summary).lower():
        print(a.headline); print(a.summary); print()

3 Space Stocks To Watch With Up To 28% Revenue Growth
SpaceX’s plan to extend Starlink into direct mobile service has turned low Earth orbit into prime economic real estate. Suddenly, the companies building rockets, satellites, and ground systems look less like science projects and more like critical digital infrastructure. That shift matters for investors who want exposure to this new layer of connectivity. This article walks through three stocks from a global space infrastructure shortlist that are worth a closer look. The stocks covered below...

'This SpaceX supplier has a front-row seat to major shifts in warfare and rocket use'- CNBC Pro
https://www.cnbc.com/2026/10/02/applied-aerospace-investors-need-to-watch-shares-of-this-spacex-supplier.html?msockid=21c662101c496b42343370ae1de86aaa



In [4]:
from finagent.agent.loop import make_client
from finagent.workflows.news_ingest import collect_articles
from finagent.workflows.news_classify import make_llm_classifier, classify_articles
from finagent.workflows.news_specialists import make_llm_analyzer, build_insights
from finagent.workflows.news_ingest import fetch_articles

client = make_client()
clf = make_llm_classifier(client)

for t, name, subject in (
    ("NVDA", "NVIDIA Corporation", "NVDA (NVIDIA Corporation)"),
    ("AADX", "Applied Aerospace & Defense, Inc.",
     "AADX (Applied Aerospace & Defense, Inc.)"),
):
    batch = collect_articles(t, company_name=name)
    cls = classify_articles(batch.articles, subject, clf)
    ins = build_insights(cls, subject, make_llm_analyzer(client))
    print(f"\n{t}: candidates {len(batch.articles)}, kept {len(cls.items)}, "
          f"irrelevant {cls.irrelevant}, unchecked {cls.unchecked}, "
          f"specialist failed {ins.failed}")
    for i in ins.insights:
        print(f"  [{i['category']}/{i['sentiment']}] {i['summary']}")
    print("  dropped:", [a.headline[:50] for a in cls.dropped][:5])

    raw = fetch_articles("NVDA")
mentions = [a for a in raw if "nvidia" in (a.headline + a.summary).lower() or "nvda" in (a.headline + a.summary).lower()]
print(len(raw), "raw articles,", len(mentions), "mention Nvidia")
for a in mentions[:8]: print(" ", a.published.date(), a.headline[:90])


NVDA: candidates 20, kept 1, irrelevant 19, unchecked 0, specialist failed 0
  [market/positive] Nvidia shareholders have experienced a significant increase in their wealth over the past five years.
  dropped: ['You Bought a Semiconductor Fund. On $500,000, $87,', 'Coca-Cola vs. PepsiCo: Which Dividend King Deserve', 'SpaceX Stock Has Climbed About 50% Through 6 Insid', 'If You Own SPY, This Lower-Cost Fund Tracks Identi', "Prediction: Here's What a $5,000 Investment in Arc"]

AADX: candidates 4, kept 2, irrelevant 2, unchecked 0, specialist failed 0
  [general/positive] Jim Cramer recommends sticking with Applied Aerospace, favoring it alongside Nokia and BlackBerry, while questioning Ambarella and expressing a dislike for food stocks.
  [general/positive] AADX is a supplier for SpaceX and is positioned to benefit from shifts in warfare and rocket use, with investors advised to watch AADX shares.
  dropped: ['3 Space Stocks To Watch With Up To 28% Revenue Gro', 'DEFSEC Technologies s

In [5]:
from finagent.workflows.news_ingest import fetch_articles
raw = fetch_articles("NVDA")
print(len(raw), min(a.published for a in raw).date(), max(a.published for a in raw).date())

249 2026-10-12 2026-10-13


In [1]:
from finagent.agent.loop import make_client
from finagent.workflows.news_ingest import collect_articles, fetch_articles
from finagent.workflows.news_classify import make_llm_classifier, classify_articles
from finagent.workflows.news_specialists import make_llm_analyzer, build_insights

client = make_client()
clf = make_llm_classifier(client)

# 1) how far back does Finnhub's response go?
raw = fetch_articles("NVDA")
print("NVDA raw:", len(raw), "articles from",
      min(a.published for a in raw).date(), "to", max(a.published for a in raw).date())

# 2) full pipeline
for t, name, subject in (
    ("NVDA", "NVIDIA Corporation", "NVDA (NVIDIA Corporation)"),
    ("AADX", "Applied Aerospace & Defense, Inc.",
     "AADX (Applied Aerospace & Defense, Inc.)"),
):
    batch = collect_articles(t, company_name=name)
    cls = classify_articles(batch.articles, subject, clf)
    ins = build_insights(cls, subject, make_llm_analyzer(client))
    print(f"\n{t}: window {batch.window_days}d widened={batch.widened}, "
          f"candidates {len(batch.articles)}, kept {len(cls.items)}, "
          f"irrelevant {cls.irrelevant}, unchecked {cls.unchecked}, "
          f"classify failed {cls.failed}, specialist failed {ins.failed}")
    for i in ins.insights:
        print(f"  [{i['category']}/{i['sentiment']}] {i['summary']}")
    print("  dropped:", [a.headline[:50] for a in cls.dropped][:5])

NVDA raw: 249 articles from 2026-10-12 to 2026-10-13

NVDA: window 14d widened=False, candidates 20, kept 8, irrelevant 5, unchecked 7, classify failed 0, specialist failed 0
  [market/positive] Nvidia shareholders have experienced a significant increase in their wealth over the past five years.
  [general/positive] Dan Ives is bullish on Nvidia Corporation and has named it as one of his top tech picks for 2027, believing that investors are underestimating a significant spending wave.
  [general/positive] Nvidia Corp has agreed to purchase stakes in four French technology startups, as reported by French newspaper Les Echos on Tuesday.
  [general/positive] Nvidia Corporation is rated a Hold, remains fundamentally strong, and is considered undervalued.
  [general/positive] Nvidia will take a minority stake in French medical website Doctolib to enhance health care through technology, as announced by CEO Jensen Huang in Paris.
  [general/positive] Navitas Semiconductor announced a new 24 k

In [1]:
from dotenv import load_dotenv; load_dotenv()
from finagent.tools.news import get_news_insights

for t in ("NVDA", "AADX"):
    r = get_news_insights(t)
    print(t, len(r))
    for i in r[:3]:
        print(" ", i["published"], i["category"], i["sentiment"], "|", i["summary"][:90])

NVDA 8
  2026-10-14 general positive | NVIDIA is collaborating with various partners to integrate its technology into AI data cen
  2026-10-14 general positive | Compal is accelerating time-to-AI using NVIDIA platforms at the OCP Global Summit 2026, fo
  2026-10-13 market positive | Nvidia shareholders have experienced a significant increase in their wealth over the last 
AADX 3
  2026-10-13 earnings positive | Applied Aerospace & Defense expects mid-teens organic revenue growth in 2027 and 2028, wit
  2026-10-13 general positive | Jim Cramer recommends sticking with Applied Aerospace, favoring it alongside Nokia and Bla
  2026-10-02 general positive | AADX is a supplier for SpaceX and is involved in significant changes in warfare and rocket


In [8]:
import inspect
import finagent.workflows.news_ingest as m
print(m.__file__)
print("has the fix:", "everything" in inspect.getsource(m.select_articles))

/Users/sudhakaransrinivasan/Documents/GitHub/multi-agent-financial-analyst/src/finagent/workflows/news_ingest.py
has the fix: True


In [6]:
t = yf.Ticker("NVDA")
info = t.info
print(info)

for key in ["freeCashflow","operatingCashflow","ttm_cashflow"]:
    print(key, "->", info.get(key, "<MISSING>"))

{'address1': '2788 San Tomas Expressway', 'city': 'Santa Clara', 'state': 'CA', 'zip': '95051', 'country': 'United States', 'phone': '408 486 2000', 'website': 'https://www.nvidia.com', 'industry': 'Semiconductors', 'industryKey': 'semiconductors', 'industryDisp': 'Semiconductors', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': "NVIDIA Corporation operates as a data center scale AI infrastructure company in the United States, Taiwan, China, Hong Kong, Europe, and internationally. It operates through Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are us

In [7]:
t = yf.Ticker("NVDA")
print(t.ttm_cashflow.loc[["Operating Cash Flow", "Capital Expenditure", "Free Cash Flow"]])

                       2026-07-31
Operating Cash Flow  1.343600e+11
Capital Expenditure -7.354000e+09
Free Cash Flow       1.270060e+11


In [1]:
from finagent.tools.fundamentals import get_fundamentals
r = get_fundamentals("NVDA")
print(r["balance_sheet"]["freeCashflow"], r["warnings"])

127006000000.0 []


In [5]:
t = yf.Ticker("NVDA")
info = t.info
print(info)
print(len(info), "fields")
for key in ["longName", "sector", "industry", "marketCap", "trailingEps"]:
    print(key, "->", info.get(key, "<MISSING>"))
print(t.calendar)

{'address1': '2788 San Tomas Expressway', 'city': 'Santa Clara', 'state': 'CA', 'zip': '95051', 'country': 'United States', 'phone': '408 486 2000', 'website': 'https://www.nvidia.com', 'industry': 'Semiconductors', 'industryKey': 'semiconductors', 'industryDisp': 'Semiconductors', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': "NVIDIA Corporation operates as a data center scale AI infrastructure company in the United States, Taiwan, China, Hong Kong, Europe, and internationally. It operates through Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are us

In [7]:
import yfinance as yf

FIELDS = ["longName", "quoteType", "sector", "industry", "marketCap",
          "trailingEps", "forwardEps", "isEarningsDateEstimate"]

for symbol in ["SPY", "ZZZZZZ", "YOUR_BIOTECH"]:
    print("TICKER", symbol)
    try:
        t = yf.Ticker(symbol)
        info = t.info
        print("fields:", len(info))
        for key in FIELDS:
            print(f"  {key}: {info.get(key, '<MISSING>')!r}")
        print("  calendar:", t.calendar)
    except Exception as exc:
        print("  ERROR:", type(exc).__name__, exc)

TICKER SPY
fields: 104
  longName: 'State Street SPDR S&P 500 ETF Trust'
  quoteType: 'ETF'
  sector: '<MISSING>'
  industry: '<MISSING>'
  marketCap: '<MISSING>'
  trailingEps: '<MISSING>'
  forwardEps: '<MISSING>'
  isEarningsDateEstimate: '<MISSING>'


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SPY"}}}


  calendar: {}
TICKER ZZZZZZ


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ZZZZZZ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ZZZZZZ"}}}


fields: 1
  longName: '<MISSING>'
  quoteType: '<MISSING>'
  sector: '<MISSING>'
  industry: '<MISSING>'
  marketCap: '<MISSING>'
  trailingEps: '<MISSING>'
  forwardEps: '<MISSING>'
  isEarningsDateEstimate: '<MISSING>'
  calendar: {}
TICKER YOUR_BIOTECH


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: YOUR_BIOTECH"}}}


fields: 1
  longName: '<MISSING>'
  quoteType: '<MISSING>'
  sector: '<MISSING>'
  industry: '<MISSING>'
  marketCap: '<MISSING>'
  trailingEps: '<MISSING>'
  forwardEps: '<MISSING>'
  isEarningsDateEstimate: '<MISSING>'


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: YOUR_BIOTECH"}}}


  calendar: {}


In [10]:
%pip install finnhub-python

Note: you may need to restart the kernel to use updated packages.


In [34]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("FINNHUB_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


In [ ]:
import finnhub

# Setup client
finnhub_client = finnhub.Client(api_key="")




# Earnings surprises
print(finnhub_client.company_earnings('NVDA', limit=10))

[{'symbol': 'NVDA', 'estimate': 2.1384, 'actual': 2.22, 'period': '2026-09-30', 'surprise': 0.0816, 'surprisePercent': 3.8159, 'year': 2027, 'quarter': 2}, {'symbol': 'NVDA', 'estimate': 1.7922, 'actual': 1.87, 'period': '2026-06-30', 'surprise': 0.0778, 'surprisePercent': 4.341, 'year': 2027, 'quarter': 1}, {'symbol': 'NVDA', 'estimate': 1.5634, 'actual': 1.62, 'period': '2026-03-31', 'surprise': 0.0566, 'surprisePercent': 3.6203, 'year': 2026, 'quarter': 4}, {'symbol': 'NVDA', 'estimate': 1.2746, 'actual': 1.3, 'period': '2025-12-31', 'surprise': 0.0254, 'surprisePercent': 1.9928, 'year': 2026, 'quarter': 3}]


In [ ]:

from openrouter import OpenRouter



client = OpenRouter(api_key="$OPENROUTER_API_KEY")



resp = client.chat.completions.create(

  model="openrouter/auto-beta",

  messages=[{"role": "user", "content": "Hello, OpenRouter."}],

)

In [26]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 25.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai]2m1/2 [openai]


In [28]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


In [29]:


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

MODEL = "openai/gpt-4o-mini"  # pinned on purpose; confirm the ID on openrouter.ai/models

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
)
print(response.choices[0].message.content)

ready


In [32]:
models = client.models.list()

for model in models.data[:20]:
    print(model.id)

google/gemini-nano-banana-2.1
mistralai/mistral-large-4-0
inclusionai/ling-3.1-flash
apodex/apodex-1.1-mini:free
unbiased/pareto-26.10-preview
openai/gpt-6.1-sol-pro
openai/gpt-6.1-sol
anthropic/claude-sonnet-5.5
anthropic/claude-sonnet-5.5:batch
typesafe/jev-router
perceptron/perceptron-mk1.5
fireworks/ember-1
z-ai/glm-5.3-prime
qwen/qwen3.8-max-prime
aion-labs/aion-3.5-mini
aion-labs/aion-3.5
upstage/solar-mini4
cohere/command-a-plus
openai/gpt-6-luna-pro
openai/gpt-6-luna-pro:batch


In [33]:
models = client.models.list()

matches = [
    model.id
    for model in models.data
    if "jev" in model.id.lower()
]

for model_id in sorted(matches):
    print(model_id)

typesafe/jev-router


In [39]:
import json


def get_pe_ratio(ticker: str) -> dict:
    """Fake tool for the plumbing test; the real one will call yfinance."""
    return {"ticker": ticker, "pe_ratio": 30.2}


tool_call = choice.message.tool_calls[0]
args = json.loads(tool_call.function.arguments)  # str -> dict
result = get_pe_ratio(args["ticker"])

messages.append(choice.message)  # the model's request
messages.append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,  # which request this answers
        "content": json.dumps(result),  # tool content must be text
    }
)

final = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
print(final.choices[0].message.content)

NVDA’s trailing P/E is **30.2**. That alone doesn’t tell us whether it’s expensive: valuation depends on expected earnings growth, risk, and how it compares with peers and its own history.


### Function : Get Company Profile

In [44]:

##basically pick a stock and then also  include the peers
## get their fundamentals which are not text 
## a Sub Agent should assess than 

print(finnhub_client.company_basic_financials('NVDA', 'all'))
# Company Peers
print(finnhub_client.company_peers('NVDA'))






{'metric': {'10DayAverageTradingVolume': 107.92138, '13WeekPriceReturnDaily': 22.1682, '26WeekPriceReturnDaily': 36.9839, '3MonthADReturnStd': 38.053997, '3MonthAverageTradingVolume': 134.69213, '52WeekHigh': 240.0983, '52WeekHighDate': '2026-10-05', '52WeekLow': 164.27, '52WeekLowDate': '2026-03-30', '52WeekPriceReturnDaily': 27.3318, '5DayPriceReturnDaily': 5.145, 'assetTurnoverAnnual': 1.0442, 'assetTurnoverTTM': 1.2788, 'beta': 2.2586465, 'bookValuePerShareAnnual': 6.4719, 'bookValuePerShareQuarterly': 9.4829, 'bookValueShareGrowth5Y': 56.87, 'capexCagr5Y': 39.89, 'cashFlowPerShareAnnual': 3.9778, 'cashFlowPerShareQuarterly': 5.2597, 'cashFlowPerShareTTM': 8.20683, 'cashPerSharePerShareAnnual': 2.5739, 'cashPerSharePerShareQuarterly': 4.1152, 'currentDividendYieldTTM': 0.1163, 'currentEv/freeCashFlowAnnual': 60.381, 'currentEv/freeCashFlowTTM': 45.9615, 'currentRatioAnnual': 3.9053, 'currentRatioQuarterly': 4.5889, 'dividendGrowthRate5Y': 20.05, 'dividendIndicatedAnnual': 1, 'divid

## News Pipeline

In [53]:
news = finnhub_client.company_news(
    "NVDA",
    _from="2026-01-01",
    to="2026-10-10"
)

In [ ]:
print(news[0])


{'category': 'company', 'datetime': 1791352200, 'headline': 'How Investing in This 1 Cryptocurrency Could Make You a Millionaire', 'id': 142679093, 'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png', 'related': 'NVDA', 'source': 'Yahoo', 'summary': 'With the wildly bullish growth outlook for AI, Venice Token is skyrocketing in value.', 'url': 'https://finnhub.io/api/news?id=36a2275de9eea24ccb9d9f77567380d970b5bdb4faa2b8c7d68bea2aeaf11d22'}


In [50]:
headline = news[0]["headline"]
summary = news[0]["summary"]

In [52]:
print(summary)

With the wildly bullish growth outlook for AI, Venice Token is skyrocketing in value.


In [54]:
import json

def classify_news(headline: str, summary: str) -> dict:
    prompt = f"""
You are a financial news analyst.

Classify the following news article about a publicly traded company.

Headline:
{headline}

Summary:
{summary}

Return ONLY valid JSON with this structure:

{{
    "sentiment": "positive | negative | neutral",
    "category": "earnings | product | management | regulation | macroeconomic | merger_acquisition | legal | analyst_rating | other",
    "relevance": "high | medium | low",
    "reason": "one short sentence"
}}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    result = response.choices[0].message.content

    return json.loads(result)

In [59]:
article = news[100]

classification = classify_news(
    headline=article["headline"],
    summary=article["summary"]
)

print(classification)

{'sentiment': 'neutral', 'category': 'product', 'relevance': 'medium', 'reason': 'The article discusses a technical controversy regarding chip usage without indicating a clear positive or negative impact on the companies.'}


# Tool Calling Checks

In [1]:
from finagent.tools.company_profile import get_company_profile as g

p = g("NVDA")
print(p)

Profile(ticker='NVDA', name='NVIDIA Corporation', quote_type='EQUITY', sector='Technology', industry='Semiconductors', market_cap=5734188187648.0, size_bucket='mega', is_profitable=True, is_pre_revenue=False, next_earnings_date=datetime.date(2026, 11, 17), days_to_earnings=41, warnings=[])


In [2]:
from finagent.tools.fundamentals import get_fundamentals
print(get_fundamentals("KO")["sentiment"]["dividendYield"])

0.0246


In [17]:
from finagent.tools.fundamentals import get_fundamentals
print(get_fundamentals("NVDA"))

{'ticker': 'NVDA', 'valuation': {'trailingPE': 28.986094, 'forwardPE': 14.406897, 'priceToBook': 24.178003, 'priceToSalesTrailing12Months': 18.273836, 'enterpriseToEbitda': 28.318, 'pegRatio': 0.29}, 'profitability': {'trailingEps': 7.91, 'grossMargins': 0.74674004, 'operatingMargins': 0.66236997, 'profitMargins': 0.63663, 'returnOnEquity': 1.17211}, 'growth': {'revenueGrowth': 1.059, 'earningsGrowth': 1.278}, 'balance_sheet': {'debtToEquity': 0.16971, 'currentRatio': 4.589, 'freeCashflow': 41809874944.0, 'totalCash': 62469001216.0, 'totalDebt': 38860001280.0}, 'price': {'currentPrice': 229.28, 'fiftyTwoWeekHigh': 243.37, 'fiftyTwoWeekLow': 164.27, 'fiftyDayAverage': 221.3926, 'twoHundredDayAverage': 201.9261}, 'sentiment': {'targetMeanPrice': 328.71695, 'numberOfAnalystOpinions': 59.0, 'dividendYield': 0.0042}, 'derived': {'net_cash': 23608999936.0, 'fcf_yield': 0.007551783234552327, 'pe_recomputed': 28.986093552465235}, 'gaps': [], 'warnings': []}


In [7]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))   # searches upward from the notebook's folder to the repo root

True

In [8]:
  from finagent.tools.earnings_history import get_earnings_history
  print(get_earnings_history("NVDA")["summary"])

{'n': 4, 'beats': 4, 'in_line': 0, 'misses': 0, 'beat_rate': 1.0, 'avg_surprise_pct': 3.442516816021538, 'latest_surprise_pct': 3.8159371492704985}


In [12]:
  from finagent.tools.earnings_history import get_earnings_history
  print(get_earnings_history("APP")["summary"])

{'n': 4, 'beats': 3, 'in_line': 1, 'misses': 0, 'beat_rate': 0.75, 'avg_surprise_pct': 3.5336915026264375, 'latest_surprise_pct': 0.06919678501090418}
